[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/ema-geometric.ipynb)

# Unrolling the EMA: why the recurrence is a geometric series

The exponential moving average (EMA) is usually introduced as a recurrence: keep a running estimate, and at each step nudge it toward the newest observation $x_t$ by blending the old estimate with the new sample,

$$\mathrm{EMA}_t \leftarrow \beta\,\mathrm{EMA}_{t-1} + (1-\beta)x_t.$$

Here $\beta \in [0,1)$ controls the memory of the average: values close to $1$ (like $0.99$) forget slowly and produce a smooth, heavily-averaged estimate, while values close to $0$ track the latest data almost immediately. Written this way, the EMA looks like a purely sequential, one-step-at-a-time computation — it is not obvious what it is actually averaging, or over how much history.

The recurrence has an equivalent, non-recursive form. If you start from $\mathrm{EMA}_0 = 0$ and repeatedly substitute the recurrence into itself, each past sample $x_{t-k}$ ($k$ steps before the present) ends up multiplied by its own fixed coefficient, and the whole average collapses into a single weighted sum:

$$\mathrm{EMA}_t = \sum_{k=0}^{t-1} (1-\beta)\beta^k \, x_{t-k}.$$

In words: the most recent sample ($k=0$) gets weight $1-\beta$, the sample before it ($k=1$) gets weight $(1-\beta)\beta$, the one before that gets $(1-\beta)\beta^2$, and so on — each step further into the past is discounted by another factor of $\beta$. That is exactly the defining property of a geometric series: a fixed ratio $\beta$ between consecutive terms.

This identity matters because geometric series have a well-known closed form for their partial sums. Summing just the *weights* (ignoring the data values $x_{t-k}$ they multiply) gives

$$\sum_{k=0}^{t-1} (1-\beta)\beta^k = 1-\beta^t,$$

which is *not* $1$ for finite $t$. Early on — small $t$ — the weights the recurrence has actually applied so far add up to only $1-\beta^t$, not the full $1$ a true average's weights should sum to; the missing mass $\beta^t$ is exactly the portion of the strip that has not been "cut" yet, in the geometric-series picture used in the accompanying interactive demo. That shortfall is precisely why $\mathrm{EMA}_t$ systematically underestimates the signal at small $t$ (it starts at $0$ no matter what the data looks like), and it is exactly what *bias correction* fixes, by rescaling the raw estimate back up to what a properly-weighted average would give:

$$\widehat{\mathrm{EMA}}_t = \frac{\mathrm{EMA}_t}{1-\beta^t}.$$

As $t\to\infty$, $\beta^t \to 0$, the correction factor $\to 1$, and $\mathrm{EMA}_t$ behaves like an ordinary (if geometrically-weighted) moving average.

This notebook computes the recurrence and the closed-form weighted sum side by side on the same synthetic data and checks numerically that they agree to floating-point precision, verifies the geometric-series identity for the weights, and plots the geometric decay of the weights themselves.

In [ ]:
#@title Setup: data generator + plotting helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def make_stream(n, seed=0):
    """A synthetic noisy signal, oldest sample first (index 0 = oldest)."""
    rng = np.random.RandomState(seed)
    trend = np.linspace(1.0, 2.0, n)
    noise = rng.normal(0, 0.3, n)
    return trend + noise

def plot_weights(beta, n):
    """Bar chart of the geometric weights (1-beta)*beta**k, k=0 (most recent) first."""
    k = np.arange(n)
    w = (1 - beta) * beta**k
    plt.figure(figsize=(7, 3))
    plt.bar(k, w, color='#3a78c9')
    plt.xlabel('k steps back from the present')
    plt.ylabel('weight  (1-β)β^k')
    plt.title(f'Geometric weights for β={beta}  (sum = {w.sum():.4f}, vs 1-β^{n} = {1-beta**n:.4f})')
    plt.tight_layout()
    plt.show()

In [ ]:
beta = 0.9
n = 30
x = make_stream(n)  # x[0] is oldest, x[n-1] is the most recent sample

In [ ]:
# Form 1: the plain recurrence, applied oldest-to-newest.
ema = 0.0
for x_t in x:
    ema = beta * ema + (1 - beta) * x_t

print('recurrence EMA_t   =', ema)

In [ ]:
# Form 2: the closed-form weighted sum, indexed k steps back from the present (k=0 = most recent).
x_back = x[::-1]  # x_back[k] is the sample k steps before the present
weighted_sum = sum((1 - beta) * beta**k * x_back[k] for k in range(n))

print('closed-form sum    =', weighted_sum)
assert np.isclose(ema, weighted_sum), 'the two forms should match to floating-point precision'
print('the two forms match.')

In [ ]:
# The weights alone are a geometric series: they sum to 1 - beta**t, not 1.
weight_sum = sum((1 - beta) * beta**k for k in range(n))
print('sum of weights      =', weight_sum)
print('1 - beta**n          =', 1 - beta**n)
print('(this shortfall is exactly what bias correction divides out: EMA_t / (1-beta**t))')

In [ ]:
plot_weights(beta, n)